# SO-101 XOX · Round 1 · SmolVLA training (Colab A100)

Trains 20,000 steps from `lerobot/smolvla_base` on the pilot data (`fport/tic-tac-toe-so101-x-round1-clean-v1`, 45 episodes, 9 tasks) and uploads the model to Hugging Face.

The goal is not success: it is to see the recording → Hub → Colab → Mac chain work. Plan: `docs/plan.md`, Step 4.

**Before you start**

1. Runtime → Change runtime type → **A100 GPU**.
2. The key icon on the left (Secrets) → add a Hugging Face token **with write access** named `HF_TOKEN` and turn on "Notebook access". Do not paste the token into a cell.
3. Run the cells in order. Training takes about 4 hours (3 h 45 min measured on an 80 GB A100); keep the tab open.

Every 5,000 steps a checkpoint is uploaded to the Hub. If the connection drops, the checkpoints up to that point are not lost.

## 1 · Setup

In [ ]:
%pip install -q "lerobot[dataset,smolvla,training]==0.6.1"

## 2 · Hugging Face login and GPU check

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import HfApi, login

HF_TOKEN = (userdata.get("HF_TOKEN") or "").strip()
assert HF_TOKEN, "Add HF_TOKEN to Secrets and turn on notebook access."
login(token=HF_TOKEN, add_to_git_credential=False)
print("Hugging Face user:", HfApi().whoami()["name"])

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > A100 GPU."
gpu = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu} ({gpu_gb:.0f} GB)")
if gpu_gb < 35:
    print("WARNING: This notebook is tuned for an A100 (40 GB). On a smaller GPU lower BATCH_SIZE (T4/L4: 16).")

## 3 · Settings and dataset check

The camera names are mapped to the names SmolVLA expects: `top` → `camera1`, `wrist` → `camera2`. There is no third camera, hence `empty_cameras=1`.

In [ ]:
import json
from datetime import datetime, timezone
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

DATASET_REPO = "fport/tic-tac-toe-so101-x-round1-clean-v1"
MODEL_REPO = "fport/smolvla-tic-tac-toe-so101-x-round1-20k"
BASE_MODEL = "lerobot/smolvla_base"
RENAME_MAP = {
    "observation.images.top": "observation.images.camera1",
    "observation.images.wrist": "observation.images.camera2",
}
EMPTY_CAMERAS = 1
BATCH_SIZE = 64
STEPS = 20_000
SAVE_FREQ = 5_000

meta = LeRobotDatasetMetadata(DATASET_REPO)
tasks = sorted(meta.tasks.index)
print("episodes:", meta.total_episodes, "| frames:", meta.total_frames, "| fps:", meta.fps)
print("cameras:", sorted(meta.video_keys))
print("tasks:", *tasks, sep="\n  ")
assert meta.total_episodes == 45, meta.total_episodes
assert sorted(meta.video_keys) == sorted(RENAME_MAP), meta.video_keys
assert len(tasks) == 9, tasks

## 4 · Training

The setting LeRobot recommends for SmolVLA: only the action expert is trained, the vision-language model is frozen (the `train_expert_only` and `freeze_vision_encoder` defaults). A log line is printed every 100 steps; `loss` should go down over time.

In [ ]:
import subprocess

run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
output_dir = f"/content/outputs/{MODEL_REPO.split('/')[-1]}_{run_id}"
command = [
    "lerobot-train",
    f"--policy.path={BASE_MODEL}",
    f"--dataset.repo_id={DATASET_REPO}",
    f"--policy.empty_cameras={EMPTY_CAMERAS}",
    f"--rename_map={json.dumps(RENAME_MAP, separators=(',', ':'))}",
    f"--policy.repo_id={MODEL_REPO}",
    "--policy.private=true",
    "--policy.push_to_hub=true",
    "--policy.device=cuda",
    f"--batch_size={BATCH_SIZE}",
    f"--steps={STEPS}",
    f"--save_freq={SAVE_FREQ}",
    "--save_checkpoint_to_hub=true",
    "--log_freq=100",
    "--num_workers=8",
    f"--output_dir={output_dir}",
    f"--job_name={MODEL_REPO.split('/')[-1]}",
    "--wandb.enable=false",
]
print(" \\\n  ".join(command), "\n")

process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
assert process.wait() == 0, f"Training ended with an error (exit code {process.returncode}). Copy the last lines and send them."
print("\nTraining finished.")

## 5 · Check the model on the Hub

In [ ]:
api = HfApi()
files = api.list_repo_files(MODEL_REPO, repo_type="model")
root_files = [f for f in files if "/" not in f]
checkpoints = sorted({f.split("/")[1] for f in files if f.startswith("checkpoints/")})
print("root files:", root_files)
print("checkpoints:", checkpoints)
assert "model.safetensors" in root_files, "The final model is not on the Hub."
print(f"\nModel ready: https://huggingface.co/{MODEL_REPO}")